In [23]:
import torch, datasets, faiss, chromadb
from sentence_transformers import SentenceTransformer

print("✅ All libraries imported successfully!")
print("GPU Name:", torch.cuda.get_device_name(0))


✅ All libraries imported successfully!
GPU Name: NVIDIA GeForce MX250


In [24]:
from datasets import load_dataset
dt = load_dataset("openai/openai_humaneval")
print(dt)

DatasetDict({
    test: Dataset({
        features: ['task_id', 'prompt', 'canonical_solution', 'test', 'entry_point'],
        num_rows: 164
    })
})


In [25]:
import datasets
print(datasets.__file__)


d:\Anaconda\envs\tf-gpu-fix\lib\site-packages\datasets\__init__.py


In [26]:
dt

DatasetDict({
    test: Dataset({
        features: ['task_id', 'prompt', 'canonical_solution', 'test', 'entry_point'],
        num_rows: 164
    })
})

In [27]:
metadata = [
    {
        "canonical_solution": rec["canonical_solution"],
        "task_id": rec["task_id"],
    }
    for rec in dt["test"]
]


In [28]:
metadata[10]

{'canonical_solution': "    if not string:\n        return ''\n\n    beginning_of_suffix = 0\n\n    while not is_palindrome(string[beginning_of_suffix:]):\n        beginning_of_suffix += 1\n\n    return string + string[:beginning_of_suffix][::-1]\n",
 'task_id': 'HumanEval/10'}

In [29]:
doc_prompt = dt["test"]["prompt"]

In [30]:
doc_prompt[10]

'\n\ndef is_palindrome(string: str) -> bool:\n    """ Test if given string is a palindrome """\n    return string == string[::-1]\n\n\ndef make_palindrome(string: str) -> str:\n    """ Find the shortest palindrome that begins with a supplied string.\n    Algorithm idea is simple:\n    - Find the longest postfix of supplied string that is a palindrome.\n    - Append to the end of the string reverse of a string prefix that comes before the palindromic suffix.\n    >>> make_palindrome(\'\')\n    \'\'\n    >>> make_palindrome(\'cat\')\n    \'catac\'\n    >>> make_palindrome(\'cata\')\n    \'catac\'\n    """\n'

In [31]:
import re

def chunk_text(text, max_length=512, overlap=50):
    """
    تقسيم النص إلى مقاطع (chunks) بطول محدد مع تداخل جزئي بين المقاطع.
    
    Args:
        text (str): النص الأصلي.
        max_length (int): الحد الأقصى لعدد الكلمات في كل مقطع.
        overlap (int): عدد الكلمات المشتركة بين كل مقطع والذي يليه.
    
    Returns:
        list: قائمة بالمقاطع الناتجة.
    """
    # تنظيف النص من المسافات الزائدة
    text = re.sub(r'\s+', ' ', text.strip())
    
    words = text.split()
    chunks = []
    
    start = 0
    while start < len(words):
        end = start + max_length
        chunk = ' '.join(words[start:end])
        chunks.append(chunk)
        
        # نقل البداية للأمام مع مراعاة التداخل
        start += max_length - overlap
    
    return chunks


# مثال على الاستخدام
data = """
Artificial Intelligence (AI) is a branch of computer science that aims to create 
machines capable of performing tasks that typically require human intelligence, 
such as reasoning, learning, and problem-solving.
"""

chunks = chunk_text(data, max_length=30, overlap=5)
for i, c in enumerate(chunks):
    print(f"Chunk {i+1}:\n{c}\n")


Chunk 1:
Artificial Intelligence (AI) is a branch of computer science that aims to create machines capable of performing tasks that typically require human intelligence, such as reasoning, learning, and problem-solving.

Chunk 2:
reasoning, learning, and problem-solving.



In [32]:
import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))


Torch version: 2.4.1+cu121
CUDA available: True
GPU Name: NVIDIA GeForce MX250


In [33]:
!pip install huggingface-hub==0.4.1


ERROR: Ignored the following yanked versions: 0.8.0, 0.9.0.dev0, 0.9.0rc0, 0.16.1, 0.26.4, 0.31.3
ERROR: Could not find a version that satisfies the requirement huggingface-hub==0.4.1 (from versions: 0.0.1, 0.0.2, 0.0.3rc1, 0.0.3rc2, 0.0.5, 0.0.6, 0.0.7, 0.0.8, 0.0.9, 0.0.10, 0.0.11, 0.0.12, 0.0.13, 0.0.14, 0.0.15, 0.0.16, 0.0.17, 0.0.18, 0.0.19, 0.1.0, 0.1.1, 0.1.2, 0.2.0, 0.2.1, 0.4.0, 0.5.0, 0.5.1, 0.6.0rc0, 0.6.0, 0.7.0rc0, 0.7.0, 0.8.0rc0, 0.8.0rc1, 0.8.0rc2, 0.8.0rc3, 0.8.0rc4, 0.8.1, 0.9.0rc2, 0.9.0rc3, 0.9.0, 0.9.1, 0.10.0rc0, 0.10.0rc1, 0.10.0rc3, 0.10.0, 0.10.1, 0.11.0rc0, 0.11.0rc1, 0.11.0, 0.11.1, 0.12.0rc0, 0.12.0, 0.12.1, 0.13.0rc0, 0.13.0rc1, 0.13.0, 0.13.1, 0.13.2, 0.13.3, 0.13.4, 0.14.0rc0, 0.14.0rc1, 0.14.0, 0.14.1, 0.15.0rc0, 0.15.0, 0.15.1, 0.16.0rc0, 0.16.2, 0.16.3, 0.16.4, 0.17.0rc0, 0.17.0, 0.17.1, 0.17.2, 0.17.3, 0.18.0rc0, 0.18.0, 0.19.0rc0, 0.19.0, 0.19.1, 0.19.2, 0.19.3, 0.19.4, 0.20.0rc0, 0.20.0rc1, 0.20.0, 0.20.1, 0.20.2, 0.20.3, 0.21.0rc0, 0.21.0, 0.21.1, 

In [34]:
!pip show huggingface-hub


Name: huggingface-hub
Version: 0.35.3
Summary: Client library to download and publish models, datasets and other repos on the huggingface.co hub
Home-page: https://github.com/huggingface/huggingface_hub
Author: Hugging Face, Inc.
Author-email: julien@huggingface.co
License: Apache
Location: d:\anaconda\envs\tf-gpu-fix\lib\site-packages
Requires: filelock, fsspec, packaging, pyyaml, requests, tqdm, typing-extensions
Required-by: accelerate, datasets, sentence-transformers, tokenizers, transformers


In [35]:
from sentence_transformers import SentenceTransformer
import torch

device = "cpu" 
model_id = "sentence-transformers/all-MiniLM-L6-v2"  # موديل embeddings
model = SentenceTransformer(model_id, device=device)

dim = model.get_sentence_embedding_dimension()
print("Embedding dimension:", dim)


Embedding dimension: 384


In [36]:
#encoded_docs = model.encode(doc_texts, show_progress_bar=True)
encoded_docs = model.encode(
    doc_prompt,            
    show_progress_bar=True,
    convert_to_tensor=True,     
    device=device
).to(torch.float32)

Batches: 100%|██████████| 6/6 [00:06<00:00,  1.08s/it]


In [37]:
encoded_docs.shape

torch.Size([164, 384])

In [38]:
doc_prompt[10], encoded_docs[10]

('\n\ndef is_palindrome(string: str) -> bool:\n    """ Test if given string is a palindrome """\n    return string == string[::-1]\n\n\ndef make_palindrome(string: str) -> str:\n    """ Find the shortest palindrome that begins with a supplied string.\n    Algorithm idea is simple:\n    - Find the longest postfix of supplied string that is a palindrome.\n    - Append to the end of the string reverse of a string prefix that comes before the palindromic suffix.\n    >>> make_palindrome(\'\')\n    \'\'\n    >>> make_palindrome(\'cat\')\n    \'catac\'\n    >>> make_palindrome(\'cata\')\n    \'catac\'\n    """\n',
 tensor([-3.4713e-02,  2.6085e-02,  2.4460e-02, -5.7956e-02, -1.4657e-01,
          7.3733e-03, -2.9972e-02,  4.0060e-02, -7.0756e-02, -1.7878e-02,
          1.3731e-02,  5.2727e-02, -1.1513e-02, -1.0795e-02, -7.5890e-02,
         -2.5067e-02, -2.9130e-02,  2.0343e-02,  7.7241e-04, -5.2517e-02,
          4.9004e-02, -2.6684e-03, -4.1855e-02,  4.3837e-02,  2.3905e-02,
          3.20

Chroma Database

In [39]:
import chromadb
chromadb_client = chromadb.PersistentClient(path=r"D:\NLP\codes\Task3\task3_chroma_db")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [40]:
collection = chromadb_client.get_or_create_collection(
    name ="humaneval",
    metadata = {"hnsw:space": "cosine"}
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [41]:
embeddings = encoded_docs.detach().cpu().numpy().astype("float32").tolist()
task_ids   = [str(x["task_id"]) for x in dt["test"]]

collection.add(
    documents=doc_prompt,     # list of prompts
    embeddings=embeddings,    # embeddings as float32 list
    metadatas=metadata,       # [{"prompt":..., "canonical_solution":..., "task_id":...}, ...]
    ids=task_ids              # list of strings
)


Add of existing embedding ID: HumanEval/0
Add of existing embedding ID: HumanEval/1
Add of existing embedding ID: HumanEval/2
Add of existing embedding ID: HumanEval/3
Add of existing embedding ID: HumanEval/4
Add of existing embedding ID: HumanEval/5
Add of existing embedding ID: HumanEval/6
Add of existing embedding ID: HumanEval/7
Add of existing embedding ID: HumanEval/8
Add of existing embedding ID: HumanEval/9
Add of existing embedding ID: HumanEval/10
Add of existing embedding ID: HumanEval/11
Add of existing embedding ID: HumanEval/12
Add of existing embedding ID: HumanEval/13
Add of existing embedding ID: HumanEval/14
Add of existing embedding ID: HumanEval/15
Add of existing embedding ID: HumanEval/16
Add of existing embedding ID: HumanEval/17
Add of existing embedding ID: HumanEval/18
Add of existing embedding ID: HumanEval/19
Add of existing embedding ID: HumanEval/20
Add of existing embedding ID: HumanEval/21
Add of existing embedding ID: HumanEval/22
Add of existing embed

In [42]:
prompt = "Write a Python function that checks if a number is prime"
prompt_embed = model.encode(prompt, convert_to_numpy=True).astype("float32")

results = collection.query(
    query_embeddings=[prompt_embed.tolist()],   # خليها list جوه list
    n_results=3
)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


In [43]:
results

{'ids': [['HumanEval/39', 'HumanEval/75', 'HumanEval/96']],
 'distances': [[0.2957614064216614, 0.2977384328842163, 0.30469346046447754]],
 'metadatas': [[{'canonical_solution': '    import math\n\n    def is_prime(p):\n        if p < 2:\n            return False\n        for k in range(2, min(int(math.sqrt(p)) + 1, p - 1)):\n            if p % k == 0:\n                return False\n        return True\n    f = [0, 1]\n    while True:\n        f.append(f[-1] + f[-2])\n        if is_prime(f[-1]):\n            n -= 1\n        if n == 0:\n            return f[-1]\n',
    'task_id': 'HumanEval/39'},
   {'canonical_solution': '    def is_prime(n):\n        for j in range(2,n):\n            if n%j == 0:\n                return False\n        return True\n\n    for i in range(2,101):\n        if not is_prime(i): continue\n        for j in range(2,101):\n            if not is_prime(j): continue\n            for k in range(2,101):\n                if not is_prime(k): continue\n                i

Faiss vector database

In [44]:
import faiss
import numpy as np
from copy import deepcopy

In [45]:
norm_encoded_docs = encoded_docs.detach().cpu().numpy().astype('float32')
faiss.normalize_L2(norm_encoded_docs)

In [46]:
faiss_index = faiss.IndexIDMap(faiss.IndexFlatIP(dim))
int_ids = np.arange(len(norm_encoded_docs))
faiss_index.add_with_ids(norm_encoded_docs, int_ids)

In [47]:
prompt = "Write a Python function that checks if a number is prime"
prompt_embed = model.encode([prompt], convert_to_numpy=True).astype("float32")

faiss.normalize_L2(prompt_embed)

results= faiss_index.search(prompt_embed, k=3)

results

(array([[0.70505166, 0.66997755, 0.6685506 ]], dtype=float32),
 array([[31, 75, 82]], dtype=int64))

In [48]:
import pickle

with open("faiss_index.pkl", "wb") as f:
    # write faiss index and document texts to the pickle file
    pickle.dump(faiss_index, f, protocol=pickle.HIGHEST_PROTOCOL)
    pickle.dump(doc_prompt, f)



with open("data.pkl", "wb") as handle:
   pickle.dump({
       "data":doc_prompt,
       "docs_id":task_ids,
        "metadata":metadata,
   },handle, protocol=pickle.HIGHEST_PROTOCOL)

In [49]:
import time

In [50]:
t0= time.process_time()

for i in range(len(doc_prompt)):

    ques = encoded_docs[i]

    results = collection.query(
        query_embeddings=ques.tolist(),
        n_results=3
    )

print("chromaDB Time:", len(doc_prompt), "prompts")

print("Time:", time.process_time() - t0)

chromaDB Time: 164 prompts
Time: 0.65625


In [51]:
encoded_docs = encoded_docs.detach().cpu().numpy().astype("float32")

In [52]:
t0= time.process_time()

for i in range(len(doc_prompt)):

    ques = encoded_docs[i].reshape(1, dim)
    faiss.normalize_L2(ques)

    results= faiss_index.search(ques, 3)

print("Faiss:", len(doc_prompt), "prompts")
print("Time:", time.process_time() - t0)    

Faiss: 164 prompts
Time: 2.65625


In [53]:
t0 = time.process_time()

queries = encoded_docs.copy()
faiss.normalize_L2(queries)

results = faiss_index.search(queries, 3)

print("Faiss:", len(doc_prompt), "prompts")
print("Time:", time.process_time() - t0)


Faiss: 164 prompts
Time: 0.046875


Accuracy

In [54]:
chroma_results = []
for i in range(len(doc_prompt)):

    ques = encoded_docs[i]

    results = collection.query(
        query_embeddings=ques.tolist(),
        n_results=3
    )
    chroma_results.append(results)

In [55]:
chroma_insights = {
    "valid": 0,
    "similar": 0,
    "invalid": 0
}

for i in range(len(doc_prompt)):
    true_id = task_ids[i]
    pred_id = chroma_results[i]["ids"][0][0]

    true_canonical = metadata[i]['canonical_solution']

    # 👇 بدل int(pred_id)
    pred_index = task_ids.index(pred_id)
    pred_canonical = metadata[pred_index]['canonical_solution']

    if str(true_id) == str(pred_id):
        chroma_insights["valid"] += 1

    elif true_canonical == pred_canonical:
        chroma_insights["similar"] += 1

    else:
        chroma_insights["invalid"] += 1

# حساب النسب المئوية
total = len(doc_prompt)
chroma_insights["valid_percentage"] = chroma_insights["valid"] / total * 100
chroma_insights["similar_percentage"] = chroma_insights["similar"] / total * 100
chroma_insights["invalid_percentage"] = chroma_insights["invalid"] / total * 100  

print("Model Id:", model_id)
print("valid:", chroma_insights["valid"])
print("valid_percentage: {:.2f}%".format(chroma_insights["valid_percentage"]))
print("similar:", chroma_insights["similar"])
print("similar_percentage: {:.2f}%".format(chroma_insights["similar_percentage"]))
print("invalid:", chroma_insights["invalid"])
print("invalid_percentage: {:.2f}%".format(chroma_insights["invalid_percentage"]))


Model Id: sentence-transformers/all-MiniLM-L6-v2
valid: 80
valid_percentage: 48.78%
similar: 0
similar_percentage: 0.00%
invalid: 84
invalid_percentage: 51.22%


In [56]:
faiss_results = []

for i in range(len(doc_prompt)):

    ques = encoded_docs[i].reshape(1, dim)
    faiss.normalize_L2(ques)

    scores, ids = faiss_index.search(ques,3)

    faiss_results.append ({
        "scores":scores,
        "ids":ids
    })


In [57]:
faiss_insights = {
    "valid": 0,
    "similar": 0,
    "invalid": 0
}

total = len(doc_prompt)

for i in range(total):
    # الـ ID الحقيقي للمسألة الحالية
    true_id = task_ids[i]
    true_canonical = metadata[i]['canonical_solution']

    # الـ ID المتوقع من FAISS (رقمي)
    pred_id = faiss_results[i]["ids"][0][0]
    pred_index = int(pred_id)  # ← مهم لو numpy.int64
    pred_canonical = metadata[pred_index]['canonical_solution']

    # 👇 المقارنة
    if true_id == task_ids[pred_index]:
        faiss_insights["valid"] += 1

    elif true_canonical == pred_canonical:
        faiss_insights["similar"] += 1

    else:
        faiss_insights["invalid"] += 1

# حساب النسب
faiss_insights["valid_percentage"] = faiss_insights["valid"] / total * 100
faiss_insights["similar_percentage"] = faiss_insights["similar"] / total * 100
faiss_insights["invalid_percentage"] = faiss_insights["invalid"] / total * 100

print(f"\n🔸 Retrieval Evaluation (FAISS) for {model_id}")
print(f"  Valid:   {faiss_insights['valid']} / {total} ({faiss_insights['valid_percentage']:.2f}%)")
print(f"  Similar: {faiss_insights['similar']} / {total} ({faiss_insights['similar_percentage']:.2f}%)")
print(f"  Invalid: {faiss_insights['invalid']} / {total} ({faiss_insights['invalid_percentage']:.2f}%)")



🔸 Retrieval Evaluation (FAISS) for sentence-transformers/all-MiniLM-L6-v2
  Valid:   164 / 164 (100.00%)
  Similar: 0 / 164 (0.00%)
  Invalid: 0 / 164 (0.00%)


In [58]:
import time
import numpy as np

# ---------- CHROMA SPEED + ACCURACY ----------
t0 = time.process_time()
chroma_results = collection.query(
    query_embeddings=encoded_docs.tolist(),
    n_results=3
)
chroma_time = time.process_time() - t0

# Evaluate Recall@1 و Recall@3
correct_ids = task_ids
chroma_recall1 = 0
chroma_recall3 = 0
for i, true_id in enumerate(correct_ids):
    retrieved = chroma_results["ids"][i]   # top-3 IDs
    if true_id == retrieved[0]:
        chroma_recall1 += 1
    if true_id in retrieved:
        chroma_recall3 += 1

chroma_recall1 /= len(correct_ids)
chroma_recall3 /= len(correct_ids)


# ---------- FAISS SPEED + ACCURACY ----------
t0 = time.process_time()
faiss_D, faiss_I = faiss_index.search(norm_encoded_docs, 3)
faiss_time = time.process_time() - t0

faiss_recall1 = 0
faiss_recall3 = 0
for i, true_id in enumerate(correct_ids):
    retrieved = [task_ids[idx] for idx in faiss_I[i]]
    if true_id == retrieved[0]:
        faiss_recall1 += 1
    if true_id in retrieved:
        faiss_recall3 += 1

faiss_recall1 /= len(correct_ids)
faiss_recall3 /= len(correct_ids)


# ---------- PRINT RESULTS ----------
print("===== ChromaDB =====")
print(f"Time: {chroma_time:.4f} sec")
print(f"Recall@1: {chroma_recall1:.4f}")
print(f"Recall@3: {chroma_recall3:.4f}")

print("\n===== FAISS =====")
print(f"Time: {faiss_time:.4f} sec")
print(f"Recall@1: {faiss_recall1:.4f}")
print(f"Recall@3: {faiss_recall3:.4f}")


===== ChromaDB =====
Time: 0.6875 sec
Recall@1: 0.4878
Recall@3: 0.6280

===== FAISS =====
Time: 0.0000 sec
Recall@1: 1.0000
Recall@3: 1.0000


In [59]:
def retrieve_from_faiss(user_query, k=3):
    # 1. تحويل الاستعلام إلى embedding
    q_emb = model.encode([user_query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(q_emb)

    # 2. البحث في FAISS
    D, I = faiss_index.search(q_emb, k)

    # 3. تنظيم النتائج
    ctx = []
    for rank, idx in enumerate(I[0]):
        ctx.append({
            "task_id": task_ids[idx],
            "prompt": doc_prompt[idx],
            "solution": metadata[idx]['canonical_solution'],  # ✅ هنا التعديل
            "score": float(D[0][rank])
        })
    return ctx


In [60]:
context = retrieve_from_faiss("Write a function that checks if a number is prime", k=3)
for c in context:
    print(c["task_id"], c["prompt"][:50], "...")


HumanEval/31 

def is_prime(n):
    """Return true if a given n ...
HumanEval/75 
def is_multiply_prime(a):
    """Write a function ...
HumanEval/82 
def prime_length(string):
    """Write a function ...


In [61]:
def build_prompt(user_query, context):
    examples = ""
    for i, ex in enumerate(context, 1):
        examples += f"\n### Example {i}\n# Task:\n{ex['prompt']}\n# Reference Solution:\n{ex['solution']}\n"

    system = (
        "You are a Python coding assistant.\n"
        "Given some examples of tasks and their solutions, "
        "write correct Python code for the new task.\n"
        "Return ONLY Python code without explanations.\n"
    )

    return f"{system}\n\n### New Task\n{user_query}\n\n### Similar Examples\n{examples}\n\n### Final Answer:\n"


In [64]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch, os

# نخزن الملفات على D بدل C
os.environ["HF_HOME"] = r"D:\NLP\codes\Task3"

gen_model_id = "Salesforce/codegen-350M-mono"

tok = AutoTokenizer.from_pretrained(gen_model_id, cache_dir=os.environ["HF_HOME"])
gen_model = AutoModelForCausalLM.from_pretrained(
     gen_model_id,
    torch_dtype=torch.float32,   # CPU version
    device_map="auto",
    cache_dir=os.environ["HF_HOME"],
    offload_folder=r"D:\NLP\codes\Task3\offload", 
    trust_remote_code=True,
)

generator = pipeline(
    "text-generation", 
    model=gen_model, 
    tokenizer=tok
)

print("✅ Model ready to generate!")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Some parameters are on the meta device because they were offloaded to the disk and cpu.


✅ Model ready to generate!


In [65]:
def generate_code(user_task, topk=1, max_new_tokens=200):
    ctx = retrieve_from_faiss(user_task, k=topk)

    # هنا نكتب prompt مخصص يضمن إن الموديل يطبع الكود بس
    prompt = f"""
You are an AI code assistant.
I will give you a programming task description and some similar examples.
Please output ONLY the final Python function implementation. 
Do not include explanations or extra text.
Do NOT repeat functions.

Please output ONLY ONE final Python function that solves the task below.
Do NOT output multiple functions.
Do NOT repeat the context.
Do NOT include explanations or docstrings.

Task: {user_task}

Context examples:
{ctx}

### Final Answer:
"""

    out = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=True, temperature=0.2, top_p=0.95
    )[0]["generated_text"]

    code = out.split("### Final Answer:")[-1].strip()
    return code, ctx

In [ ]:
user_task = "Write a function that checks if a number is prime."
gen_code, ctx = generate_code(user_task, topk=1, max_new_tokens=120)

with open(r"D:\NLP\codes\Task3\code5.py", "w", encoding="utf-8") as f:
    f.write(gen_code)

print("✅ Code saved to code5.py")


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


✅ Code saved to code1.py


In [67]:
def evaluate_output(generated_code, ctx):
    reference = ctx[0]["solution"]
    print("=== Generated Code ===")
    print(generated_code[:300], "\n")
    print("=== Reference Solution ===")
    print(reference[:300], "\n")

    # مقياس بدائي: هل أول سطر (تعريف الدالة) متطابق
    return generated_code.splitlines()[0].strip() == reference.splitlines()[0].strip()


In [68]:
import traceback

def run_tests_on_generated(task_id, generated_code):
    rec = next(r for r in dt["test"] if r["task_id"] == task_id)
    test_code = rec["test"]

    local_env = {}
    try:
        exec(generated_code, {}, local_env)  # نفّذ الكود المولّد
        exec(test_code, {}, local_env)       # نفّذ اختبارات HumanEval للمهمة
        return True, "All tests passed ✅"
    except Exception as e:
        return False, f"Test failed ❌: {e}\n{traceback.format_exc()}"

